In [1]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
from numpy.ma.core import less_equal


In [2]:
# Create Model
m = gp.Model("Model_1")

Set parameter Username
Set parameter LicenseID to value 2739971
Academic license - for non-commercial use only - expires 2026-11-17


In [3]:
# Time Based GMCNF parameters

# Gravitational acceleration [m/sˆ2]
g_0 = 9.80665

# Number of nodes i,j
nodes = 4

Connections = {0: [0,1] ,
               1: [0,1,2],
               2: [1,2,3],
               3: [2,3]}

# Time Steps 12 (days) #testing with +1 day
T = 12

# Advanced Time window
T_adv = list(range(T))
print(T_adv)
#in case multiple time windows are needed this can be added


#Node Open Windows:
#Arcs can only depart or arrive at these nodes at the times specified (So always include the start and end of the window in the nodes)
#This makes every single arc unique based on its departing time and arrival node
#So: holding arc become multipliers for whatever time can be kept
N_Window = {0: [0,4,8,9,10,11],
             1:[0,5,9,10,11],
               2: T_adv,
                3:[0,2,3,4,5,6,11]}

reverseN_window = {key:list(reversed(item)) for key,item in N_Window.items()}


# Velocity change [km/StructureMass]
#this model optimizes for IMLEO,
#  so it does not consider any velocity chage necessary for LEO-PAC (splashdown)
#However, since the model includes splash down, and we want only a single launch per day,
#We can add a Large (big PropCapacity ) cost for PAC to LEO to ensure that the model does not use this arc
# unless it is really necessary

#Pacific Ocean, Low Earth Orbit, Lunar Lunar Orbit, Lunar Surface
# PAC, LEO, LLO, LS are 0, 1, 2, 3
delta_V = {0: {0: 0, 1: 1000}, # PAC to LEO is Big PropCapacity high
            1: {0: 0, 1: 0, 2: 4.04},
              2: {1: 4.04, 2: 0, 3: 1.87},
                3: {2: 1.87, 3: 0}}

# Time of travel [days]
TOF = {0: {0: 1, 1: 1},
        1: {0: 1, 1: 1, 2: 3},
          2: {1: 3, 2: 1, 3: 1},
            3: {2: 1, 3: 1}}

ReverseTOF = {a:{} for a in TOF.keys()}
for a,b in TOF.items():
  for b1,c1 in b.items():
    ReverseTOF[a][b1] = -c1


#Shows off all possible arcs
#In order [starttime][startnode][endnode]{"ArrivalTime", "FullTravelTime"}
def AllpossibleOutflowArcs(Connections, T_adv, window =N_Window, TOFused = TOF):
  

  AllArcs = {}

  for t in T_adv:
    TimeNode = {}
    for i in Connections:
      if t in window[i]:
        
        Now = window[i].index(t)
        TimeNode[i] = {}
        
        for j in Connections[i]:
            
            if t+TOFused[i][j] in window[j]:
              TimeNode[i][j]= {"ArrivalTime": t+TOFused[i][j], "FullTravelTime":TOFused[i][j] }

        #If the holding arc is not available, then the arc to the next available
        #free time block is added, as long as we are not at the end of the window (N_Window[i])
        if (i not in TimeNode[i]) and (Now+1 != len(window[i])): 
          TimeNode[i][i]={"ArrivalTime":window[i][Now+1],"FullTravelTime":window[i][Now+1] - t}
        
        
    if TimeNode != {}:
      AllArcs[t] = TimeNode
            
                    
    
  return AllArcs
AllArcs = AllpossibleOutflowArcs(Connections,T_adv,window=N_Window,TOFused = TOF)


RevAllArcs = AllpossibleOutflowArcs(Connections,T_adv,window=reverseN_window,TOFused = ReverseTOF)
           
print(Connections)    
print(len(AllArcs))   
print(AllArcs[1]) 
print(AllArcs[11])
                
        
        


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
{0: [0, 1], 1: [0, 1, 2], 2: [1, 2, 3], 3: [2, 3]}
12
{2: {2: {'ArrivalTime': 2, 'FullTravelTime': 1}, 3: {'ArrivalTime': 2, 'FullTravelTime': 1}}}
{0: {}, 1: {}, 2: {}, 3: {}}


In [4]:
"""
# Number of vehicle types
V = 2



Y = GRB.INTEGER
# Spacecrafts of same type


#Vehicles: [Saturn V stage 2, Saturn V stage 3, Command Module, Service Module, LM Descent Stage, LM ascent stage]


# Structure mass [kg]
StructureMass = np.array([2500, 30000])

# Specific impulses [StructureMass]
I_sp = np.array([200, 500])

# Payload Capacity [kg]
PayloadCap = np.array([1000,75])

# Propellant Capacity [kg] 
PropCapacity = np.array([65000, 1700000])

"""

'\n# Number of vehicle types\nV = 2\n\n\n\nY = GRB.INTEGER\n# Spacecrafts of same type\n\n\n#Vehicles: [Saturn V stage 2, Saturn V stage 3, Command Module, Service Module, LM Descent Stage, LM ascent stage]\n\n\n# Structure mass [kg]\nStructureMass = np.array([2500, 30000])\n\n# Specific impulses [StructureMass]\nI_sp = np.array([200, 500])\n\n# Payload Capacity [kg]\nPayloadCap = np.array([1000,75])\n\n# Propellant Capacity [kg] \nPropCapacity = np.array([65000, 1700000])\n\n'

In [5]:
#"""

#payload system testing!

# Number of vehicle types
V = 2



Y = GRB.INTEGER
# Spacecrafts of same type


#Vehicles: [Saturn V stage 2, Saturn V stage 3, Command Module, Service Module, LM Descent Stage, LM ascent stage]


# Structure mass [kg]
StructureMass = np.array([2500, 30])

# Specific impulses [s]
I_sp = np.array([900, 200])

# Payload Capacity [kg]
PayloadCap = np.array([10000,75])

# Propellant Capacity [kg] 
PropCapacity = np.array([4000, 17000])


#"""

In [6]:
"""
#Simplified test data


# Number of vehicle types
V = 4



Y = GRB.INTEGER
# Spacecrafts of same type


# Structure mass [kg]
StructureMass = np.array([40000, 15000,3000,255 ])

# Specific impulses [StructureMass]
I_sp = np.array([421, 324,0,0])

# Payload Capacity [kg]
PayloadCap = np.array([5000, 2500,200,12])

# Propellant Capacity [kg]
PropCapacity = np.array([1200770, 400000,0,0])

"""


'\n#Simplified test data\n\n\n# Number of vehicle types\nV = 4\n\n\n\nY = GRB.INTEGER\n# Spacecrafts of same type\n\n\n# Structure mass [kg]\nStructureMass = np.array([40000, 15000,3000,255 ])\n\n# Specific impulses [StructureMass]\nI_sp = np.array([421, 324,0,0])\n\n# Payload Capacity [kg]\nPayloadCap = np.array([5000, 2500,200,12])\n\n# Propellant Capacity [kg]\nPropCapacity = np.array([1200770, 400000,0,0])\n\n'

In [7]:
# VEHICLE DATA

"""






# Number of vehicle types
V = 6



Y = GRB.INTEGER
# Spacecrafts of same type


#Vehicles: [Saturn V stage 2, Saturn V stage 3, Command Module, Service Module, LM Descent Stage, LM ascent stage]


# Structure mass [kg]
StructureMass = np.array([38415, 12014, 4841, 6053, 2770, 1719])

# Specific impulses [StructureMass]
I_sp = np.array([421, 421, 0, 314, 311, 311])

# Payload Capacity [kg]
PayloadCap = np.array([0, 0, 524, 60, 500, 250])

# Propellant Capacity [kg] 
PropCapacity = np.array([452045, 107725, 0, 18413, 8804, 2358])




#"""




'\n\n\n\n\n\n\n# Number of vehicle types\nV = 6\n\n\n\nY = GRB.INTEGER\n# Spacecrafts of same type\n\n\n#Vehicles: [Saturn V stage 2, Saturn V stage 3, Command Module, Service Module, LM Descent Stage, LM ascent stage]\n\n\n# Structure mass [kg]\nStructureMass = np.array([38415, 12014, 4841, 6053, 2770, 1719])\n\n# Specific impulses [StructureMass]\nI_sp = np.array([421, 421, 0, 314, 311, 311])\n\n# Payload Capacity [kg]\nPayloadCap = np.array([0, 0, 524, 60, 500, 250])\n\n# Propellant Capacity [kg] \nPropCapacity = np.array([452045, 107725, 0, 18413, 8804, 2358])\n\n\n\n\n#'

In [8]:
# COMMODITY Data and Demand/Supply

# Propellant mass fraction
#defined from rocket equation 1-e**(-deltav/Ispg0)

#actually the official function is e**(-deltav/Ispg0), but using the 1-e form allows use 
# to multiply the contents with the unchanging masses to include their input into the transformation linearly
#This varies with the delta v necessary for each arc, and the Isp of the vehicle used for that arc
#Will be used later


def phi(i,j,v, dV = delta_V, I_sp = I_sp, g_0 = g_0):
    if I_sp[v] == 0:
        return 1
    else:
        return 1 - np.exp(-(1000*dV[i][j] / (I_sp[v] * g_0))) #1000 used for conversion



# Commodity variable types
# Crew, consumables kg, equipment kg, samples kg, propellant kg
X = [GRB.INTEGER, GRB.CONTINUOUS, GRB.CONTINUOUS, GRB.CONTINUOUS, GRB.CONTINUOUS]

# Crew mass [kg/crew]
crew_mass = 100


CommodityMassConversion = [crew_mass,1,1,1,1] #what to multiply the commodity with to get Kg
PropIndex = 4

#In a separate list:
#Add a variable for each  rocket, listed in Carriable to then recognize the vehicle
Carriable = {} #index of vehicle

CarriedVar = [] # Variable used for the payload
for i,x in enumerate(I_sp):
    Carriable[i] = GRB.INTEGER
    #Carriable.append(i)
    CarriedVar.append(GRB.INTEGER) #add an integer variable to the commodity vector

# Crew, consumables kg, equipment kg, samples kg, propellant kg
# PAYLOAD ASSUMPTIONS

# Consumption rates [kg/crew/day]
food_consumption = 1.0
water_consumption = 5.0
oxygen_consumption = 1.1
consumption = food_consumption + water_consumption + oxygen_consumption
#if the model works, this can be made more granular by separating the consumptions




# Upper limit of each variable per spacecraft is the capacity of each spacecraft *number of spacecraft in that node.
# Crew, consumables kg, equipment kg, samples kg, propellant kg
XUpper = [PayloadCap/crew_mass, PayloadCap, PayloadCap, PayloadCap, PropCapacity]

#Single Spacecraft consumption table (from outflow to inflow consumption of all commodities)

#Matrix multiplication with a vector of outflows


# Propellant usage Matrix
def Solo_SC_Consumption(i, j,v, consumption = consumption, TOF = TOF,structure_mass = StructureMass, extraPayload = Carriable,PropellantIndex = PropIndex):


    NumbComm = 5 #5 Commodities 
    extraNumb = len(extraPayload)

    CommodityBlocklen = NumbComm+1

    Full_Length =  NumbComm + extraNumb + 1
    FullMatrix = np.zeros((Full_Length,Full_Length))

    # Crew, consumables kg, equipment kg, samples kg, propellant kg, number of spacecraft

    
    CommodityBlock = np.array([[1, 0, 0, 0, 0, 0], #Crew
                        [-consumption*TOF[i][j] , 1, 0, 0, 0, 0], # Consumable consumption
                        [0, 0, 1, 0, 0, 0], #Equipment
                        [0, 0, 0, 1, 0, 0], #Samples
                        [crew_mass * -1*phi(i,j,v,delta_V,I_sp,g_0),
                       -1*phi(i,j,v,delta_V,I_sp,g_0),
                         -1*phi(i,j,v,delta_V,I_sp,g_0),
                           -1*phi(i,j,v,delta_V,I_sp,g_0),
                             1-1*phi(i,j,v,delta_V,I_sp,g_0),
                                -1*structure_mass[v]*phi(i,j,v,delta_V,I_sp,g_0)], #Propellant
                        [0, 0, 0, 0, 0, 1]])# last row (and column) is for number of spacecraft
    
    FullMatrix[:CommodityBlocklen,:CommodityBlocklen] = CommodityBlock




    
    for i1,(x1,vtype) in enumerate(extraPayload.items()):
        i2 = len(extraPayload) - i1
        FullMatrix[-i2,-i2] = 1
        FullMatrix[PropellantIndex, -i2] = -1*structure_mass[x1]*phi(i,j,v,delta_V,I_sp,g_0)


    return FullMatrix


#Arc commodity transformations when there is no propellant burn
def Solo_SC_Consumption_NodV(i, j, consumption = consumption, TOF = TOF, extraPayload = Carriable):

    NumbComm = 5 #5 Commodities 
    extraNumb = len(extraPayload)

    CommodityBlocklen = NumbComm+1

    Full_Length =  NumbComm + extraNumb + 1
    FullMatrix = np.zeros((Full_Length,Full_Length))

    # Crew, consumables kg, equipment kg, samples kg, propellant kg, number of spacecraft
    CommodityBlock = np.array([[1, 0, 0, 0, 0, 0], #Crew
                        [-consumption*TOF[i][j] , 1, 0, 0, 0, 0], # Consumable consumption
                        [0, 0, 1, 0, 0, 0], #Equipment
                        [0, 0, 0, 1, 0, 0], #Samples
                        [0, 0, 0, 0, 1, 0], #Propellant
                        [0, 0, 0, 0, 0, 1]])# last row (and column) is for number of spacecraft
    
    FullMatrix[:CommodityBlocklen,:CommodityBlocklen] = CommodityBlock

    for i1,(x1) in enumerate(extraPayload.items()):
        i2 = len(extraPayload) - i1
        FullMatrix[-i2,-i2] = 1


    return FullMatrix










# THE COMMODITIES ARE PROVIDED AT LEO. START ALL THINGS AT LEO
# CHECK DAYS FOR MISSION !!!
#Commodity Array: D[Node][Day][Commodity]

#This is technically fine, sinc ewe are going through every point whether or not the window is open
#However if we want to switch to dictionaries, then the correct time has to be used
#not just the order of the sequence in the array (skipping nodes)

D = [[np.array([0 for x in range(len(X))])
      for _ in T_adv]
    for _ in Connections]

print(len(D))
print(len(D[0]))
print(len(D[0][0]))

#initial test values
# Earth (PAC) consumables, equipment, propellant supply infinite at leo time 0, AND Moon surface sample supply (infinite at all times)
#Crew are capped in supply so we don't leave anyone on the moon
D[1][0][0] = 3 #Crew
D[1][0][1] = 99999 #Consumables
D[1][0][2] = 99999 #Equipment
D[1][0][4] = 99999999 #Propellant

for x in T_adv:
    D[3][x][3] = 999999 #Moon samples


# APOLLO
# Remember we are using a list starting at 0
# Crew demand/supply
D[3][4][0] = -2 # Lunar surface day 5 crew demand (negative supply)
D[2][3][0] = -1 # Lunar orbit day 4 crew demand
D[3][5][0] = 2 # Lunar surface day 6 crew supply (return)
D[2][6][0] = 1 # Lunar orbit day 7 crew supply (return)
D[0][11][0] = -3 # Earth day 11 crew demand (return)

D[3][4][2] = -420 # Lunar surface day 5 (scientific) equipment demand

D[0][11][3] = -110 # Earth day 11 lunar sample demand





# S/PayloadCap COMMODITY DEMAND
# format d[node][vehicle][day]
# Infinite supply of SC at LEO day 1 (time 0), none elsewhere
d = [[[1 if (i == 1 and t == 0) else 0 for t in range(T)] # Infinite supply of spacecrafts at i = 1, t = 0 LEO
     for _ in range(V)]
    for i in Connections]


4
12
5


In [9]:
# CREATE COMMODITY FLOW VECTORS AND S/PayloadCap COMMODITY FLOW
# Variable naming process: commodity_{direction}flow_{v},{i},{j},{t},{x}'
#v: vehicle type, i: node of origin, j: node of destination, t: time step, x: commodity type
# Spacecraft Variable naming process: sc_commodity_{direction}flow_{v},{i},{j},{t}'




#Only create variables for arcs that end within the destination window.

#TOF tells you the travel times
#ef check_destination_window(startnode, endnode, tstart, All_nodes= T_adv, TOF = TOF):
#    arrival =  TOF[startnode][endnode]+tstart
#
#    if arrival in All_nodes:
#        return True
#    else:
#        return False


#lower bound for all commodities is 0, no negatives.
def create_commodity_flow(model, V, X, direction = "out", connect = Connections, typeC = "Classic"):

    
    
    x_flow = {v:{i:{j: {t: np.array([[model.addVar(vtype=X[x], name=f'{typeC}_commodity_{direction}flow_{v},{i},{j},Tstart{t},Tend{AllArcs[t][i][j]['ArrivalTime']},Commodity{x}',lb = 0 )]
                              for x in range(len(X))])
                    for t in AllArcs if (i in AllArcs[t]) and (j in AllArcs[t][i])} #Only make an arc if it exists in AllArcs
                for j in connect[i]}
               for i in connect}
              for v in range(V)}


    return x_flow


def create_sc_commodity_flow(model, V,Y, direction = "out", connect = Connections):
    y_flow = {v:{i:{j: {t: np.array([model.addVar(vtype=Y, name=f'sc_commodity_{direction}flow_{v},{i},{j},Tstart{t},Tend{AllArcs[t][i][j]['ArrivalTime']}',lb=0)])
            for t in AllArcs if (i in AllArcs[t]) and (j in AllArcs[t][i])}
          for j in connect[i]}
         for i in connect} for v in range(V)}

    return y_flow

# Outflow+ leaving from node i to j, inflow- arriving at node j from i

x_outflow, x_inflow = create_commodity_flow(m, V, X, direction="out", connect=Connections), create_commodity_flow(m, V, X, direction="in", connect=Connections)
y_outflow, y_inflow = create_sc_commodity_flow(m, V, Y, direction="out", connect=Connections), create_sc_commodity_flow(m, V, Y, direction="in", connect=Connections)






def NoSelfPayload(model,V,Flowlist,Arclist = AllArcs, PayloadV = Carriable):

   

    #print(PayloadV)
    # if v is a vehicle in payloadv

    for v in range(V):
        for t in Arclist:
            for i in Arclist[t]:
                for j in Arclist[t][i]:
        #for i in connect:
        #    for j in connect[i]:
        #        for t in Time:
        #            if (t in AllArcs) and (i in AllArcs[t]) and (j in AllArcs[t][i]):
                        if v in PayloadV.keys():
                            

                            print(v)

                            model.addConstr(Flowlist[v][i][j][t][v][0] == 0,
                                            name=f'NoSelfPayloadConstraint_vehicle{v}_startnode{i}_endnode{j}_starttime{t}_endtime{AllArcs[t][i][j]['ArrivalTime']}')
    

    return

SCpayload_Outflow = create_commodity_flow(m,V,CarriedVar,"out",Connections,"SCPayload")
SCpayload_Inflow =  create_commodity_flow(m,V,CarriedVar,"in",Connections,"SCPayload")

NoSelfPayload(m,V,SCpayload_Outflow,AllArcs,Carriable)


m.update()






0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1


In [10]:
# ADD THE CONSTRAINTS (2 & 3)
# CONSTRAINTS 2 & 3 MASS BALANCE
# Node commodity demand D vectors (positive for supply)
# sum(x[i][t]+) - sum(x[i][t]-) <= D[i][t]

import sys

#this is done for all nodes except the final day

#for i in Connections:
#    for t in T_adv: #range(T - 1)) also works for simple tests
#        if i in AllArcs[t]: #Ensuring only nodes in timewindows are being checked
for t in AllArcs:
    for i in AllArcs[t]:
        
        
        
        x_outflow_sum = sum(x_outflow[v][i][j][t] 
                            if (t in AllArcs) and (i in AllArcs[t]) and (j in AllArcs[t][i])
                            else np.array([[0] for _ in range(len(X))])  #packaged SC are dealt with differently
                            for v in range(V) 
                            for j in Connections[i])
            #)
        # On the last day there is no outflow, the if else statement ensures
        #that only the outflows for which the spacecraft has had time to arrive are counted
    

        x_inflow_sum = sum(x_inflow[v][j][i][RevAllArcs[t][i][j]["ArrivalTime"]] 
                           if (t in RevAllArcs) and (i in RevAllArcs[t]) and (j in RevAllArcs[t][i])
                           else np.array([[0] for _ in range(len(X))])
                           for v in range(V) 
                           for j in Connections[i])
        # Only count the inflows for which the spacecraft has had time to arrive 
        # (or where it has had time to depart (no negative times))


        #if t == 0:
        #    print(x_outflow_sum)
        #    print(len(x_outflow_sum))
        #    print(x_inflow_sum)
        #    print(len(x_inflow_sum[0]))
        #    sys.exit()

        for x in range(len(X)):
            #if (t in AllArcs) and (i in AllArcs[t]):
                try:
                    m.addConstr(x_outflow_sum[x][0] - x_inflow_sum[x][0] <= D[i][t][x],
                                name=f"mass_balance_x_node{i}_time{t}_comm{x}")
                    #print(x_outflow_sum[x][0])
                
                except Exception as e:
                    print(f"Error on node={i}, time={t}, commodity={x}: {e}")
                    raise
                
                #except:
                #    print("Error on constraint for node {}, time {}, commodity {}".format(i, t, x))

        print(i,t)
        # S/PayloadCap commodity supply and demand
        for v in range(V):
            y_outflow_sum = sum(y_outflow[v][i][j][t]  
                                if (t in AllArcs) and (i in AllArcs[t]) and (j in AllArcs[t][i]) \
                                else np.array([0]) 
                                for j in Connections[i])

            y_inflow_sum = sum(y_inflow[v][j][i][RevAllArcs[t][i][j]["ArrivalTime"]] 
                               if (t in RevAllArcs) and (i in RevAllArcs[t]) and (j in RevAllArcs[t][i])\
                               else np.array([0])
                               for j in Connections[i])

            
            #S/PayloadCap Payload commodity supply and demand, the payloads must be summed into the node considerations
            #so the SCpayload_Inflow and Outflow variables are summed to the exisitng SC sums. Either a ship or a payload will be added
            
            
            if  v in Carriable.keys():
                #All payload variables are added to this sum, since the self carrying constraint is already added
                # all vehicles are payloadable, this may change
                
                

                payloadSumOut = sum(SCpayload_Outflow[v1][i][j][t][v] 
                            if (t in AllArcs) and (i in AllArcs[t]) and (j in AllArcs[t][i])
                            else np.array([0]) 
                            for v1 in range(V) 
                            for j in Connections[i])
                
                y_outflow_sum = sum(y_outflow_sum, payloadSumOut)

                #if i == 1 and t == 0:
                #    print(y_outflow_sum)
                #print(payloadSumOut)
                #print(t)
                #sys.exit()
                
                payloadSumIn = sum(SCpayload_Inflow[v1][j][i][RevAllArcs[t][i][j]["ArrivalTime"]][v] 
                            if (t in RevAllArcs) and (i in RevAllArcs[t]) and (j in RevAllArcs[t][i])\
                            else np.array([0]) 
                            for v1 in range(V) 
                            for j in Connections[i])
                
                y_inflow_sum = sum(y_inflow_sum,payloadSumIn)
                

            #print(y_inflow_sum)
            #print(y_outflow_sum)

            #m.addConstr(y_outflow_sum[0] - y_inflow_sum[0] <= d[i][v][t],
            #name=f"SC_mass_balance_x_node{i}_time{t}_vehicle{v}")

            if (t < T-1) and (t != 0):

                    m.addConstr(y_outflow_sum[0] - y_inflow_sum[0] == d[i][v][t],
                    name=f"SC_lossless_mass_balance_x_node{i}_time{t}_vehicle{v}") #lose no SC until the end
            else:
                    m.addConstr(y_outflow_sum[0] - y_inflow_sum[0] <= d[i][v][t],
                    name=f"SC_mass_balance_x_node{i}_time{t}_vehicle{v}")

        
        


m.update()

0 0
1 0
2 0
3 0
2 1
2 2
3 2
2 3
3 3
0 4
2 4
3 4
1 5
2 5
3 5
2 6
3 6
2 7
0 8
2 8
0 9
1 9
2 9
0 10
1 10
2 10
0 11
1 11
2 11
3 11


In [11]:
## Commodity transformation
#import sys

for t in AllArcs:
    for i in AllArcs[t]:
        for j in AllArcs[t][i]:
#for i in Connections:
#    for j in Connections[i]:
#        for t in T_adv:

#             if check_destination_window(i,j,t,T_adv,TOF):
                  
                for v in range(V):
                
                
                    #print(i,j,t,v)
                
                
                    Vout = np.concatenate((x_outflow[v][i][j][t],
                        np.array([y_outflow[v][i][j][t]])), axis=0)


                    Vin = np.concatenate((x_inflow[v][i][j][t],
                         np.array([y_inflow[v][i][j][t]])), axis=0)
                    




                    #print(Vin)
                    #sys.exit()


                    #create the correct consumption matrix, based on deltav and travel time
                    if delta_V[i][j] <= 0: #If there is no Delta v: there is no propellant consumption
                        Consumed =Solo_SC_Consumption_NodV(i,j,consumption,TOF,Carriable)
                        #Matrix includes all payloads
                            
                    


                    #PropIndex, tells us which commodity is the propellant
                    
                    else:
                        Consumed =Solo_SC_Consumption(i,j,v,consumption,TOF,StructureMass,Carriable)
                        
                        #for each payload only the structural mass of the carried vehicle is be added, since having payload (or propellant) is the same mass and can be placed in the Carrying storage
                        #moving propellant from 1 payload SC to a working SC will be handled at nodes, and the relevant conversion constraints are also handled there
                       

                    
                    #add SC payload variables to Vin and Vout
                    for c1 in Carriable:
                        Vin = np.append(Vin,SCpayload_Inflow[v][i][j][t][c1])
                        Vout = np.append(Vout,SCpayload_Outflow[v][i][j][t][c1])
                        
                    
                    
                    #print(Consumed)
                    #print(Vin)
                    #print(Vout)
                    #sys.exit()
                    
                    

                    


                    transformed = np.dot(Consumed,Vout)
                
                
                    for i1,(enterarc,leavearc) in enumerate(zip(transformed, Vin)):
                        
                        #print(type(enterarc))
                        #print(type(leavearc))
                        m.addConstr(enterarc == leavearc,name=f'Arc_transformationConstraint_Start{i}_End{j}_Starttime{t}_Vehicle{v}_Commodity{i1}')

                    

                    
                    #one extra constraint is needed per arc, to make sure that only the fuel in the moving SC tank is used up 
                    # Since inflow follows from outflow, it can never be a difference larger than the capacity of the tank, even if the original
                    #amount of propellant is higher due to carried SC
                    #print(x_outflow[v][i][j][t][PropIndex])
                    m.addConstr(x_inflow[v][i][j][t][PropIndex][0] >= x_outflow[v][i][j][t][PropIndex][0] - PropCapacity[v])


In [12]:
# CONSTRAINTS 5 CONCURRENCY LIMITS

# Concurrency constraint matrix
# H[x+] <= e * y+ --> Payload mass and fuel in Spacecraft does not exceed maximum capacities

#Here 3 constraints must be added

#Payload (including structural mass of carried spacecraft) <= Max payload of SC
# Propellant in a Spaceship <= Max Propellant Occupancy + Extra space in payload if a SC is carried (scpayload *Capacity)
# Propellant+ payload <= Max fuel +Max payload (so:extra fuel is getting carried, but its in a tank in the payload section, so it is considered part of the payload)

import copy


def create_concurrency_constraint(connect = Connections, PropellantCommodityIndex = PropIndex,
                                   MassConversion = CommodityMassConversion,
                                     SCstructMass =StructureMass, payloadSC = Carriable): # Same for all vehicles, max payload mass
    
    #There are 3 separate capacities to keep in mind: Payload, Propellant, and Payload + Prop so 3 separate rows are made 1 for each
    Payloadrow = copy.deepcopy(MassConversion) 
    Payloadrow[PropellantCommodityIndex] = 0 #Massconversion is used for all classic commodities, then the propellant is removed

    Proprow = [0] * len(MassConversion)
    Proprow[PropellantCommodityIndex] = 1

    Combinedrow = copy.deepcopy(MassConversion)

    for i1,v1 in enumerate(payloadSC):
        Payloadrow.append(StructureMass[v1])
        Proprow.append(0)
        Combinedrow.append(StructureMass[v1])

    H = [{j: np.array([Payloadrow, #payload
                        Proprow,
                        Combinedrow]) #Propellant
           for j in connect[i]}
          for i in connect]
    return H


def create_sc_design_parameters(V, PayloadCap, PropCapacity,payloadSC =Carriable):
    e = np.zeros((V,3,1+len(payloadSC)))

    for i1, e1 in enumerate(e): #First 3 
            e[i1][0][0] = PayloadCap[i1]
            e[i1][1][0] = PropCapacity[i1]
            e[i1][2][0] = PayloadCap[i1] +PropCapacity[i1]


            for i2,c1 in enumerate(Carriable): #additional values for the payload variables
                 e[i1][1][1+i2] = PropCapacity[c1] 

    #e = [np.array([[PayloadCap[v]],
    #                [PropCapacity[v]],
    #                [PayloadCap[v]+PropCapacity[v]]]) for v in range(V)]
    
    

    return e


H = create_concurrency_constraint(Connections, PropIndex,CommodityMassConversion,StructureMass,Carriable)
e = create_sc_design_parameters(V, PayloadCap, PropCapacity,Carriable)
print(e)
print(len(e))


[[[10000.     0.     0.]
  [ 4000.  4000. 17000.]
  [14000.     0.     0.]]

 [[   75.     0.     0.]
  [17000.  4000. 17000.]
  [17075.     0.     0.]]]
2


In [13]:
# ADD THE CONSTRAINTS (5)

import sys

for v in range(V):
    for t in AllArcs:
        for i in AllArcs[t]:
            for j in AllArcs[t][i]:
    #for i in Connections:
    #    for j in Connections[i]:
    #        for t in range(T-1):
    #            if check_destination_window(i,j,t,T_adv,TOF):
                    
                    Extendedcommodity = x_outflow[v][i][j][t]
                    
                    #Extendedconstraint = np.zeros(len(Carriable)+1)
                    
                    Extendedconstraint = [y_outflow[v][i][j][t]]


                    for i1,c1 in enumerate(Carriable):
                        Extendedcommodity = np.append(Extendedcommodity,SCpayload_Outflow[v][i][j][t][c1])
                        Extendedconstraint.append(SCpayload_Outflow[v][i][j][t][c1])
                    
                    

                    for i1, (commodity, constraint) in enumerate(zip(np.dot(H[i][j],Extendedcommodity),
                                                     np.dot(e[v],Extendedconstraint))):

                        #print(commodity)
                        #print(constraint)
                        
                        m.addConstr(commodity <= constraint[0],name = f'Max_concurrency_constraint_row{i1}_vehicle{v}_startnode{i}_endnode{j}_starttime{t}')
                    #sys.exit()

m.update()


In [14]:
# CONSTRAINTS 6 TIME-WINDOW
# ADD THE CONSTRAINTS (6)

#Minimum value of 0 for all arcs, not time window

for v in range(V):
    for t in AllArcs:
        for i in AllArcs[t]:
            for j in AllArcs[t][i]:

                    for commodity_out in x_outflow[v][i][j][t]:
                        m.addConstr(commodity_out[0] >= 0)

                    for commodity_in in x_inflow[v][i][j][t]:
                        m.addConstr(commodity_in[0] >= 0)

                    m.addConstr(y_outflow[v][i][j][t][0] >= 0)
                    m.addConstr(y_inflow[v][i][j][t][0] >= 0)

m.update()

# StructureMass[v] >= 0


In [15]:
# CONSTRAINTS 7 SPACE-CRAFT MASS

In [16]:
"""
# COST FUNCTION - INITIAL MASS AT LEO
# sum(cost * x + cost_y * StructureMass * y + coststructure mass of payload SC *payloadsc)

# x = Crew, consumables, equipment, samples, propellant, crew(return)

#Currently the model assumes we are starting at the LEO node t = 0, i=1

#this function creates a matrix of 

def CostApollo(V,CostArcs, Arclist = AllArcs, massconvert = CommodityMassConversion):
    


    cost_coeff = [[{j:
                        [np.array([[crew_mass], [1], [1], [1], [1]]) if (t == 0 and i == 1)
                         else np.array([[0] for _ in range(len(X))])
                         for t in range(T-1)]
           for j in connect[i]}
          for i in connect]
         for _ in range(V)]

    sc_cost_coeff = [[{j:
                        [1 if (t == 0 and i == 1)
                         else 0
                         for t in range(T-1)]
           for j in connect[i]}
          for i in connect]
         for _ in range(V)]

    return cost_coeff, sc_cost_coeff

cost_coeff, sc_cost_coeff = create_commodity_cost(V, Connections, crew_mass)

"""

'\n# COST FUNCTION - INITIAL MASS AT LEO\n# sum(cost * x + cost_y * StructureMass * y + coststructure mass of payload SC *payloadsc)\n\n# x = Crew, consumables, equipment, samples, propellant, crew(return)\n\n#Currently the model assumes we are starting at the LEO node t = 0, i=1\n\n#this function creates a matrix of \n\ndef CostApollo(V,CostArcs, Arclist = AllArcs, massconvert = CommodityMassConversion):\n\n\n\n    cost_coeff = [[{j:\n                        [np.array([[crew_mass], [1], [1], [1], [1]]) if (t == 0 and i == 1)\n                         else np.array([[0] for _ in range(len(X))])\n                         for t in range(T-1)]\n           for j in connect[i]}\n          for i in connect]\n         for _ in range(V)]\n\n    sc_cost_coeff = [[{j:\n                        [1 if (t == 0 and i == 1)\n                         else 0\n                         for t in range(T-1)]\n           for j in connect[i]}\n          for i in connect]\n         for _ in range(V)]\n\n    re

In [17]:
# DEFINE THE COST FUNCTION (1)

"""
#General version
cost = sum(
    np.dot(cost_coeff[v][i][j][t].T, x_outflow[v][i][j][t]) + sc_cost_coeff[v][i][j][t] * StructureMass[v] * y_outflow[v][i][j][t][0]
    for v in range(V)
    for i in Connections
    for j in Connections[i]
    for t in range(3)
)
"""
#specific to apollo version
#only looking at node 1 at time t = 0

#for v1 in Carriable:
#    for v in range(V):
#        for j in AllArcs[0][1]:
#            print(v1,v,j)
#            print(SCpayload_Outflow[v][1][j][0][v1])

#Remember: AllArcs[t][i][j]  flows: [v][i][j][t]

Cost = (
    sum(
        np.dot(CommodityMassConversion, x_outflow[v][1][j][0])[0]
        + StructureMass[v] * y_outflow[v][1][j][0][0]
        for v in range(V)
        for j in AllArcs[0][1]
    )
    +
    sum(
        StructureMass[k] * SCpayload_Outflow[v][1][j][0][k][0]
        for v in range(V)
        for k in Carriable
        for j in AllArcs[0][1]
    )
)

#cost = sum(
#    np.dot(cost_coeff[v][1][j][0].T, x_outflow[v][1][j][0]) + sc_cost_coeff[v][1][j][0] * StructureMass[v] * y_outflow[v][1][j][0][0]
#    for v in range(V)
#    for j in Connections[1]


#cost = cost[0][0]
print(Cost)

m.setObjective(Cost, GRB.MINIMIZE)
m.update()

100.0 Classic_commodity_outflow_0,1,2,Tstart0,Tend3,Commodity0 + Classic_commodity_outflow_0,1,2,Tstart0,Tend3,Commodity1 + Classic_commodity_outflow_0,1,2,Tstart0,Tend3,Commodity2 + Classic_commodity_outflow_0,1,2,Tstart0,Tend3,Commodity3 + Classic_commodity_outflow_0,1,2,Tstart0,Tend3,Commodity4 + 2500.0 sc_commodity_outflow_0,1,2,Tstart0,Tend3 + 100.0 Classic_commodity_outflow_0,1,1,Tstart0,Tend5,Commodity0 + Classic_commodity_outflow_0,1,1,Tstart0,Tend5,Commodity1 + Classic_commodity_outflow_0,1,1,Tstart0,Tend5,Commodity2 + Classic_commodity_outflow_0,1,1,Tstart0,Tend5,Commodity3 + Classic_commodity_outflow_0,1,1,Tstart0,Tend5,Commodity4 + 2500.0 sc_commodity_outflow_0,1,1,Tstart0,Tend5 + 100.0 Classic_commodity_outflow_1,1,2,Tstart0,Tend3,Commodity0 + Classic_commodity_outflow_1,1,2,Tstart0,Tend3,Commodity1 + Classic_commodity_outflow_1,1,2,Tstart0,Tend3,Commodity2 + Classic_commodity_outflow_1,1,2,Tstart0,Tend3,Commodity3 + Classic_commodity_outflow_1,1,2,Tstart0,Tend3,Commodity4

In [18]:
m.optimize()

Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (mac64[arm] - Darwin 25.5.0 25F84)

CPU model: Apple M1 Pro
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 2710 rows, 1600 columns and 7000 nonzeros (Min)
Model fingerprint: 0xd783bbad
Model has 32 linear objective coefficients
Variable types: 800 continuous, 800 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e-01, 2e+04]
  Objective range  [1e+00, 2e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+08]
Presolve removed 2483 rows and 1348 columns
Presolve time: 0.02s
Presolved: 227 rows, 252 columns, 1101 nonzeros
Variable types: 172 continuous, 80 integer (25 binary)

Root relaxation: objective 1.012310e+04, 133 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 10123.0977    0   13          - 10

In [19]:
#"""
# Good practice: write the full model first
m.update()
m.write("debug_model.lp")

m.optimize()

# If Gurobi says infeasible or unbounded, resolve with DualReductions off
if m.Status == GRB.INF_OR_UNBD:
    m.Params.DualReductions = 0
    m.optimize()

if m.Status == GRB.INFEASIBLE:
    print("Model is infeasible. Computing IIS...")
    m.computeIIS()

    # Writes a small model containing the infeasible subsystem
    m.write("infeasible_subset.ilp")

    print("\nConstraints in IIS:")
    for c in m.getConstrs():
        if c.IISConstr:
            print(f"{c.ConstrName}: sense={c.Sense}, RHS={c.RHS}")

    print("\nVariable bounds in IIS:")
    for v in m.getVars():
        if v.IISLB:
            print(f"{v.VarName}: lower bound {v.LB}")
        if v.IISUB:
            print(f"{v.VarName}: upper bound {v.UB}")

#"""

Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (mac64[arm] - Darwin 25.5.0 25F84)

CPU model: Apple M1 Pro
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 2710 rows, 1600 columns and 7000 nonzeros (Min)
Model fingerprint: 0xd783bbad
Model has 32 linear objective coefficients
Variable types: 800 continuous, 800 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e-01, 2e+04]
  Objective range  [1e+00, 2e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+08]
Presolved: 227 rows, 252 columns, 1101 nonzeros

Continuing optimization...


Cutting planes:
  Gomory: 3
  Implied bound: 4
  MIR: 2
  Flow cover: 9
  Flow path: 1
  Network: 3
  Relax-and-lift: 1

Explored 1 nodes (144 simplex iterations) in 0.10 seconds (0.02 work units)
Most recent optimization runtime was 0.01 seconds (0.00 work units)
Thread count was 8 (of 8 available processors)

Solution count 4: 10701.4 10720.9 10728.9 10748.5 

Optimal solut

In [20]:
import sys
sys.path.append("/2026_code/")  # needed to add in the results file
import Results

m.update()
m.write("debug_model_Simple.lp")

Apollo_Veh =[
        "Saturn V S-II",
        "Saturn V S-IVB",
        "Command Module",
        "Service Module",
        "LM Descent Stage",
        "LM Ascent Stage"
    ]

Two_ship_test =["Payload","Fuel"]

namevar = Two_ship_test

Cargoflows,Shipflows = Results.extract_flows(
    x_outflow=x_outflow,
    x_inflow=x_inflow,
    y_outflow=y_outflow,
    arcs=Connections,
    node_names=["PAC", "LEO", "LLO", "LS"],
    vehicle_names=namevar,
    commodity_names=[
        "crew",
        "consumables",
        "equipment",
        "sample",
        "propellant"
    ],
    tof=TOF,
    T=T,
    T_adv = T_adv, 
    CommMass = CommodityMassConversion,
    StructMass= StructureMass,
    AllArcs=AllArcs,
    payloadflows=SCpayload_Outflow,
    Carryship=Carriable
)

#flows.head()


Results.plot_time_space_network(
    Shipflows,
    Cargoflows,
    node_order=["PAC", "LEO", "LLO", "LS"],
    title="Apollo 17 optimized time-space logistics solution"
)

mass_table = Results.make_mass_flow_table(Cargoflows, use="out_mass")
#mass_table

mass_table.style.format(precision=1)

Results.propellantUsage(Cargoflows)

Results.plot_vehicle_gantt(Cargoflows, title="Apollo 17 spacecraft activity")

In [21]:
mass_table.style.format(precision=1)

item,t_depart,t_arrive,from_node,to_node,vehicle,n_ships,Carried SC: Fuel,consumables,crew,equipment,propellant,sample,total_flow
0,0,3,LEO,LLO,Payload,1.0,30.0,198.8,300.0,420.0,7252.6,0.0,8201.4
1,3,4,LLO,LLO,Fuel,1.0,0.0,75.0,0.0,0.0,1738.7,0.0,1813.7
2,3,4,LLO,LS,Payload,1.0,0.0,59.9,200.0,420.0,1583.4,0.0,2263.3
3,4,5,LLO,LLO,Fuel,1.0,0.0,75.0,0.0,0.0,1738.7,0.0,1813.7
4,4,5,LS,LS,Payload,1.0,0.0,45.7,0.0,0.0,673.9,0.0,719.6
5,5,6,LLO,LLO,Fuel,1.0,0.0,75.0,0.0,0.0,1738.7,0.0,1813.7
6,5,6,LS,LLO,Payload,1.0,0.0,45.7,200.0,0.0,673.9,110.0,1029.6
7,6,7,LLO,LLO,Fuel,1.0,0.0,75.0,0.0,0.0,0.0,0.0,75.0
8,6,7,LLO,LLO,Payload,1.0,0.0,31.5,300.0,0.0,1738.7,110.0,2180.2
9,7,10,LLO,LEO,Payload,1.0,0.0,85.2,300.0,0.0,1738.7,110.0,2233.9


In [22]:
#Shipflows.head()
Cargoflows.head()

,vehicle,v,from_node,to_node,i,j,t_depart,t_arrive,cargo_type,item,quantity,out_mass,in_mass,mass_change,carried_ship_type,carried_ship_index,n_ships
0,Payload,0,LEO,PAC,1,0,10,11,commodity,crew,3.0,300.0,300.0,0.0,NaN,NaN,1.0
1,Payload,0,LEO,PAC,1,0,10,11,commodity,consumables,21.3,21.3,0.0,-21.3,NaN,NaN,1.0
2,Payload,0,LEO,PAC,1,0,10,11,commodity,sample,110.0,110.0,110.0,0.0,NaN,NaN,1.0
3,Payload,0,LEO,LLO,1,2,0,3,commodity,crew,3.0,300.0,300.0,0.0,NaN,NaN,1.0
4,Payload,0,LEO,LLO,1,2,0,3,commodity,consumables,198.8,198.8,134.9,-63.9,NaN,NaN,1.0


In [23]:
"""

# x = Crew, consumables, equipment, samples, propellant
#Variable naming process: commodity_{direction}flow_{v},{i},{j},{t},{x}'
# Variable naming process: sc_commodity_{direction}flow_{v},{i},{j},{t}'

results = {final_variable.VarName: final_variable.X for final_variable in m.getVars()}

sorted_results = dict(sorted(results.items(), key=lambda item: int(item[0].split(",")[3])))

f = open("classic_apollo_solution.txt", "w")



for final in sorted_results:
    if sorted_results[final] != 0:
        print('%StructureMass %g' % (final, sorted_results[final]))
        f.write('%StructureMass %g' % (final, sorted_results[final]))
        f.write('\n')
f.close()
"""

'\n\n# x = Crew, consumables, equipment, samples, propellant\n#Variable naming process: commodity_{direction}flow_{v},{i},{j},{t},{x}\'\n# Variable naming process: sc_commodity_{direction}flow_{v},{i},{j},{t}\'\n\nresults = {final_variable.VarName: final_variable.X for final_variable in m.getVars()}\n\nsorted_results = dict(sorted(results.items(), key=lambda item: int(item[0].split(",")[3])))\n\nf = open("classic_apollo_solution.txt", "w")\n\n\n\nfor final in sorted_results:\n    if sorted_results[final] != 0:\n        print(\'%StructureMass %g\' % (final, sorted_results[final]))\n        f.write(\'%StructureMass %g\' % (final, sorted_results[final]))\n        f.write(\'\n\')\nf.close()\n'

In [24]:
# # EQUATION 7 CONSTRAINTS
#
# # Structural Fraction (fuel dependent)
# alpha = 0.045  # LOX/kerosene
#
# # Gravitational Acceleration Earth
# g_0 = 9.8  # m/s2
#
# # Upper Bound Allowed for Propellant Tank Capacity
# M_ub = 500000  # kg
#
# # Spacecraft Impulsive Burn
# t_b = 120  # StructureMass
#
#
# # Structure Mass Variable
# def create_s_star_variables(model, v=V):
#     variables = {}
#     for v in range(V):
#         variables[v] = model.addVar(vtype=GRB.CONTINUOUS, name=f'Structure_Mass_{v}')
#     return variables
#
#
# s_star = create_s_star_variables(model=m)
#
# m.update()

In [25]:
# # CONSTRAINTS 7
#
# for v in tqdm(V):
#     m.addConstr(s_star[v] = 2.3931 * )

In [26]:
import verification as vf

report = vf.run_all_audits(globals())
vf.print_report(report)

VERIFICATION AUDIT REPORT
[     PASS]  Time windows (Eq. 6, structural)
[     PASS]  Commodity mass balance (Eq. 2)
[     PASS]  Spacecraft balance incl. carried SC (Eq. 3)
[     PASS]  Arc transformation / rocket equation (Eq. 4/8)
[     PASS]  Concurrency & capacity limits (Eq. 5)
[     PASS]  Variable domains (non-neg., integrality, no self-carry)
[     PASS]  Demand satisfaction
[     PASS]  Objective (IMLEO) recomputation (Eq. 1)
------------------------------------------------------------------------
TOTAL VIOLATIONS: 0


0